In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Optimize Shuffle Example")
    .master("local[*]")
    .config('spark.executor.cores', 4)
    .config('spark.cores.max', 16)
    .config('spark.executor.memory', "512M")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 20:40:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Check spark default Parallelism
spark.sparkContext.defaultParallelism

36

In [3]:
# Disable AQE and Broadcast join

spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")

In [5]:
_schema = "first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"

emp = spark.read.format('csv').option('header', True).schema(_schema).load('data/input/employee_records.csv')
emp.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|tiffanyjohnston@e...|       (705)900-5337|604738.0|            8|
|    Ashley|   Montoya|        C

In [6]:
# Find avg salary per dept
from pyspark.sql.functions import avg

emp_avg = emp.groupBy('department_id').agg(avg('salary')).alias('avg_salary')
emp_avg.show()

+-------------+------------------+
|department_id|       avg(salary)|
+-------------+------------------+
|            1|504876.96401242825|
|            6|504428.12590014644|
|            3| 504697.6808514883|
|            5| 504167.9429997006|
|            9| 504945.3055672206|
|            4| 505419.4963977089|
|            8| 505299.1226286386|
|            7|504514.38453985273|
|           10| 502682.2575766687|
|            2| 503563.2174529479|
+-------------+------------------+



In [8]:
from pyspark.sql.functions import expr
emp_avg_v2 = emp.groupBy('department_id').agg(expr('avg(salary) as avg_salary'))
emp_avg_v2.show()

+-------------+------------------+
|department_id|        avg_salary|
+-------------+------------------+
|            1|504876.96401242825|
|            6|504428.12590014644|
|            3| 504697.6808514883|
|            5| 504167.9429997006|
|            9| 504945.3055672206|
|            4| 505419.4963977089|
|            8| 505299.1226286386|
|            7|504514.38453985273|
|           10| 502682.2575766687|
|            2| 503563.2174529479|
+-------------+------------------+

